In [2]:
import torch
import torch.nn as nn

In [ ]:
sentence = "Life is short, eat desert first"

dc = { s:i for i,s in enumerate(sorted(sentence.replace(',', ' ').split()))}
dc

tensor([0, 4, 5, 2, 1, 3])

In [6]:
sentence_int = torch.tensor([dc[s] for s in sentence.replace(',', ' ').split()])
sentence_int

tensor([0, 4, 5, 2, 1, 3])

In [7]:
vocab_size = 50_000

torch.manual_seed(123)
embed = torch.nn.Embedding(vocab_size, 3)
embedded_sentence = embed(sentence_int).detach()

print(embedded_sentence)
print(embedded_sentence.shape)

tensor([[ 0.3374, -0.1778, -0.3035],
        [ 0.1794,  1.8951,  0.4954],
        [ 0.2692, -0.0770, -1.0205],
        [-0.2196, -0.3792,  0.7671],
        [-0.5880,  0.3486,  0.6603],
        [-1.1925,  0.6984, -1.4097]])
torch.Size([6, 3])


In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out_kq, d_out_v):
        super().__init__()
        self.d_out_kq = d_out_kq

        self.W_q = nn.Parameter(torch.rand(d_in, d_out_kq))
        self.W_k = nn.Parameter(torch.rand(d_in, d_out_kq))
        self.W_v = nn.Parameter(torch.rand(d_in, d_out_v))

    def forward(self, x):
        queries = x @ self.W_q
        keys = x @ self.W_k
        values = x @ self.W_v

        # calculating the score is not the same as letting the model use it.
        attention_scores = queries @ keys.T

        # create a mask so that the model does not calculate attention scores for future tokens.
        # torch.triu(..., diagonal=1) marks positions above the main diagonal: the future tokens.
        mask = torch.triu(torch.ones(attention_scores.shape[1], attention_scores.shape[1]),diagonal=1)
        masked = attention_scores.masked_fill(mask.bool(), -torch.inf)

        attention_weights = torch.softmax(masked / self.d_out_kq ** 0.5, dim=-1)

        context_vec = attention_weights @ values

        return context_vec
        


In [12]:
ca = CausalAttention(d_in=3, d_out_kq=2, d_out_v=4)

print(ca(embedded_sentence))


tensor([[    nan,     nan,     nan,     nan],
        [-0.1606,  0.1418, -0.1611, -0.0917],
        [ 0.8680,  0.6354,  0.4539,  0.3473],
        [ 0.5847,  0.5230,  0.2462,  0.2212],
        [ 0.3473,  0.2825,  0.1699,  0.1357],
        [-0.1597, -0.0100, -0.1415, -0.0775]], grad_fn=<MmBackward0>)
